# AF2 box-score factorial — validation only
Menguji D0FT/AF2 box × score tanpa training. Dua endpoint murni harus mereproduksi hasil historis sebelum hybrid boleh ditafsirkan. Test tidak dipulihkan atau dibaca.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/af2-box-score-factorial'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('REPO:',REPO,'| BRANCH:',BRANCH)

In [ ]:
import tarfile, torch
ARCHIVE_REL='bundles/faruq-development-v3-grouped.tar'
D0FT_REL='experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt'
AF2_REL='experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
REFERENCE_REL='experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json'
REQUIRED=(ARCHIVE_REL,D0FT_REL,AF2_REL,REFERENCE_REL)
PROJECT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE,D0FT,AF2,REFERENCE=[require_project_artifact(PROJECT,path) for path in REQUIRED]
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file(),DATA
assert not (DATA/'test').exists(),'Test tidak boleh tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'
OUTPUT_ROOT=PROJECT/'experiments/faruq-v3-af2-box-score-factorial-v1'
OUTPUT=OUTPUT_ROOT/'af2_box_score_factorial.json'
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)

In [ ]:
import json
LOG=OUTPUT_ROOT/'af2_box_score_factorial_run.log'
LOG.parent.mkdir(parents=True,exist_ok=True)
device='0' if torch.cuda.is_available() else 'cpu'
command=[sys.executable,'-u','-m','coffee_detector.analysis.af2_box_score_factorial',
 '--d0ft-checkpoint',str(D0FT),'--af2-checkpoint',str(AF2),
 '--data-root',str(DATA),'--grouped-summary',str(GROUPED),
 '--reference-summary',str(REFERENCE),'--output',str(OUTPUT),
 '--device',device,'--max-det','500','--confidence','0.001']
print('MENJALANKAN VALIDATION-ONLY FACTORIAL')
with LOG.open('w',encoding='utf-8') as stream:
    process=subprocess.run(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
if process.returncode: raise RuntimeError(f'Factorial gagal: {process.returncode}; log={LOG}')

In [ ]:
import pandas as pd
result=json.loads(OUTPUT.read_text())
labels={'DD':'D0FT box + D0FT score','DA':'D0FT box + AF2 score','AD':'AF2 box + D0FT score','AA':'AF2 box + AF2 score'}
rows=[]
for arm,metrics in result['results'].items():
    rows.append({'arm':arm,'combination':labels[arm],
                 'macro_map50_95':metrics['macro_map50_95'],
                 'bottom3_class_map50_95':metrics['bottom3_class_map50_95'],
                 'worst_class_map50_95':metrics['worst_class_map50_95'],
                 'worst_class':metrics['worst_class']})
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
print('CALIBRATION:',json.dumps(result['calibration'],indent=2))
print('GATES:',result['gates'])
print('COMPARISON:',json.dumps(result['comparison'],indent=2))
print('DECISION:',result['decision'])
print('NEXT:',result['next'])
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',OUTPUT)
print('Kirim tabel, calibration, gates, comparison, dan decision. Jangan training atau membuka test.')